In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

llm = init_chat_model("openai:gpt-4.1")

class State(TypedDict):
    messages: Annotated[list, add_messages]
    name: str
    release_date: str

In [4]:
from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.types import Command, interrupt

@tool
# 상태 업데이트를 위한 ToolMessage를 생성하는 경우,
# 일반적으로 해당 도구 호출(tool call)에 대한 ID가 필요합니다.
# 이때 LangChain의 InjectedToolCallId를 사용하면,
# 해당 인자가 도구의 스키마(schema)에 모델에게 노출되지 않도록 처리할 수 있습니다.
def human_assistance(
    name: str, release_date: str, tool_call_id: Annotated[str, InjectedToolCallId]
):
    """사람의 확인이 필요한 정보를 검토받는 도구입니다."""

    # 사람에게 정보가 맞는지 물어봅니다.
    human_response = interrupt(
        {
            "question": "이 정보가 맞나요?",
            "name": name,
            "release_date": release_date,
        }
    )

    # 사람이 "Yes"로 응답한 경우 상태를 그대로 사용
    if human_response.get("correct", "").lower().startswith("y"):
        verified_name = name
        verified_date = release_date
        response = "정보가 정확하다고 확인됨."
    # 수정이 필요한 경우, 사람이 제공한 값을 사용
    else:
        verified_name = human_response.get("name", name)
        verified_date = human_response.get("release_date", release_date)
        response = f"사람이 수정한 정보: {human_response}"

    # 상태에 name, release_date 저장하고, 메시지도 함께 반환
    state_update = {
        "name": verified_name,
        "release_date": verified_date,
        "messages": [ToolMessage(response, tool_call_id=tool_call_id)],
    }

    # 상태 갱신을 위해 Command 객체를 반환
    return Command(update=state_update)

In [5]:
search_tool = TavilySearch(max_results=2)
tools = [search_tool, human_assistance]
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    message = llm_with_tools.invoke(state["messages"])
    assert(len(message.tool_calls) <= 1)
    return {"messages": [message]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [6]:
user_input = (
    "랭그래프가 언제 출시되었는지 찾아줄래요? "
    "결과를 찾으면 human_assistance 도구를 사용해서 사람에게 확인해줘."
)

config = {"configurable": {"thread_id": "1"}}

events = graph.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    config,
    stream_mode="values",
)

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

랭그래프가 언제 출시되었는지 찾아줄래요? 결과를 찾으면 human_assistance 도구를 사용해서 사람에게 확인해줘.
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_O6YOjKogewWkzNUm1GHdFmaI)
 Call ID: call_O6YOjKogewWkzNUm1GHdFmaI
  Args:
    query: 랭그래프 출시일
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "랭그래프 출시일", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://kr.linkedin.com/pulse/langgraph-v10-coming-what-means-ai-development-zoftcares-2e5ac?tl=ko", "title": "LangGraph v1.0이 출시됩니다: 이것이 AI 개발에 의미하는 바", "content": "9월 2일, LangChain은 LangGraph v1.0의 알파 버전을 공개하며 상태 기반 AI 워크플로우 오케스트레이션에 중요한 이정표를 세웠습니다.", "score": 0.7636429, "raw_content": null, "id": "ba36dd-00"}, {"url": "https://velog.io/@hsgogo/%EB%9E%AD%EA%B7%B8%EB%9E%98%ED%94%84", "title": "랭그래프", "cont

In [7]:
snapshot = graph.get_state(config)
print(snapshot.next)

('tools',)


In [8]:
from langgraph.types import Command

human_command = Command(
    resume={
        "name": "랭그래프",
        "release_date": "2024년 1월 17일",
    },
)

events = graph.stream(human_command, config, stream_mode="values")

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_7XJxIpYNwcJsl554x9E2e9HS)
 Call ID: call_7XJxIpYNwcJsl554x9E2e9HS
  Args:
    name: 랭그래프(LangGraph) 출시일
    release_date: 2023-09-02
================================= Tool Message =================================
Name: human_assistance

사람이 수정한 정보: {'name': '랭그래프', 'release_date': '2024년 1월 17일'}
================================== Ai Message ==================================

사람의 확인을 거친 결과, 랭그래프(LangGraph)의 출시일은 2024년 1월 17일입니다.


In [9]:
snapshot = graph.get_state(config)
{k: v for k, v in snapshot.values.items() if k in ("name", "release_date")}

{'name': '랭그래프', 'release_date': '2024년 1월 17일'}

In [10]:
graph.update_state(config, {"name": "LangGraph (library)"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1d1c8d-bd8e-64bd-8006-90cd42633403'}}

In [11]:
snapshot = graph.get_state(config)
{k: v for k, v in snapshot.values.items() if k in ("name", "release_date")}

{'name': 'LangGraph (library)', 'release_date': '2024년 1월 17일'}

In [12]:
# 그래프의 상태 기록 전체를 가져옵니다 (지금까지의 실행 히스토리)
history = graph.get_state_history(config)
to_replay = None
for state in history:
    # 각 상태에서 메시지 개수와 다음 노드를 출력합니다
    print("Messages:", len(state.values["messages"]), "Next:", state.next)
    print("-" * 80)
    
    # 메시지가 6개인 상태를 선택합니다 (임의 기준으로 선택)
    if len(state.values["messages"]) == 3:
        # 나중에 되감기(replay)할 상태로 저장합니다
        to_replay = state

Messages: 6 Next: ()
--------------------------------------------------------------------------------
Messages: 6 Next: ()
--------------------------------------------------------------------------------
Messages: 5 Next: ('chatbot',)
--------------------------------------------------------------------------------
Messages: 4 Next: ('tools',)
--------------------------------------------------------------------------------
Messages: 3 Next: ('chatbot',)
--------------------------------------------------------------------------------
Messages: 2 Next: ('tools',)
--------------------------------------------------------------------------------
Messages: 1 Next: ('chatbot',)
--------------------------------------------------------------------------------
Messages: 0 Next: ('__start__',)
--------------------------------------------------------------------------------


In [15]:
print(to_replay.next)
print(to_replay.config)

('chatbot',)
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1c70-1d1d-63b2-8002-f52d9511df73'}}


In [16]:
# 저장된 시점으로 되감아 실행합니다
for event in graph.stream(None, to_replay.config, stream_mode="values"):
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================= Tool Message =================================
Name: tavily_search

{"query": "랭그래프 출시일", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://kr.linkedin.com/pulse/langgraph-v10-coming-what-means-ai-development-zoftcares-2e5ac?tl=ko", "title": "LangGraph v1.0이 출시됩니다: 이것이 AI 개발에 의미하는 바", "content": "9월 2일, LangChain은 LangGraph v1.0의 알파 버전을 공개하며 상태 기반 AI 워크플로우 오케스트레이션에 중요한 이정표를 세웠습니다.", "score": 0.7636429, "raw_content": null, "id": "ba36dd-00"}, {"url": "https://velog.io/@hsgogo/%EB%9E%AD%EA%B7%B8%EB%9E%98%ED%94%84", "title": "랭그래프", "content": "랭그래프. 현서·2025년 9월 8일. 팔로우. 1. 랭그래프. 1. 랭그래프(LangGraph). 그래프 구조 (Graph Structure). 2. 그래프. 2.1 그래프 종류. 2-2.", "score": 0.6307082, "raw_content": null, "id": "84ccab-01"}], "response_time": 3.39, "request_id": "554ffcaf-1934-4728-bae1-d7b803fa0385"}
================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_RzG1AAg08usI2